# Day 34：随机森林

使用年龄和估计薪资预测是否购买（0/1）。User ID 不作为特征。分层拆分让两部分的类别比例接近；测试集只用于最终评估。precision 表示预测正类中真阳性的比例，recall 表示实际正类被找回的比例，F1 是两者的调和平均，2PR/(P+R)。分类报告分别将每一类视为正类；macro avg 不按类别大小加权，weighted avg 按真实样本数加权。

运行前请阅读[环境与运行说明](../docs/setup.md)。本课 `.py` 是教学源文件，配套 Markdown 和 Notebook 自动同步。图形保存到 `outputs/`，设置 `COURSE_SHOW_PLOTS=1` 可显示窗口。


[Python 源文件](Day%2034_Random_Forests.py) · [Notebook](Day%2034_Random_Forests.ipynb) · [完整课程目录](../docs/curriculum.md)


In [ ]:
from pathlib import Path
import sys

# 脚本从文件位置定位仓库；Notebook 从当前工作目录向上查找。
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for candidate in (base, *base.parents):
    if (candidate / "Code" / "course_utils.py").is_file():
        code_dir = str(candidate / "Code")
        if code_dir not in sys.path:
            sys.path.insert(0, code_dir)
        break
else:
    raise FileNotFoundError("找不到课程仓库，请从仓库根目录或 Code 目录启动 Notebook。")
from course_utils import DATA, OUTPUT, finish_plot


## 数据与基线

先与始终预测训练集多数类的简单基线比较。所有类别指标必须结合样本数量解读。


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from course_utils import classification_summary, decision_plot

data = pd.read_csv(DATA / "Social_Network_Ads.csv")
X = data[["Age", "EstimatedSalary"]]
y = data["Purchased"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Majority baseline accuracy:", accuracy_score(y_test, baseline.predict(X_test)))


## 训练模型

每棵树使用 bootstrap 样本，并在节点分裂时随机选择候选特征。sklearn 平均树的类别概率后选择类别。100 棵树用于较稳定的演示，不代表最优值；树模型无需标准化。OOB 是训练过程中的辅助估计。


In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, max_features="sqrt", min_samples_leaf=2,
                               oob_score=True, random_state=0, n_jobs=1)
model.fit(X_train, y_train)
print("OOB accuracy (training estimate):", model.oob_score_)


## 测试评估与决策边界

混淆矩阵行是真实类别，列是预测类别。绘图工具在原始单位网格上调用整个模型，年龄/薪资坐标未标准化；它仅支持本课两个输入特征。


In [ ]:
y_pred = classification_summary(model, X_test, y_test, "day34")
decision_plot(model, X_test, y_test, "day34_test", ["Age (years)", "Estimated salary"])
from sklearn.inspection import permutation_importance
importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=0)
print(pd.DataFrame({"accuracy_drop_mean": importance.importances_mean,
                    "accuracy_drop_std": importance.importances_std}, index=X.columns))
# 固定模型后置乱一列，衡量测试 accuracy 的下降；仅作解释，不据此选特征或调参。


## 练习与检查

在训练交叉验证内比较 10/100 棵树的稳定性与耗时。解释 permutation importance 是预测依赖程度，不是因果效应；相关特征会影响解释。
